In [1]:
import copy
import networkx as nx
import os
import pandas as pd
import statsmodels.api as sm
import sys
import warnings

from dowhy import CausalModel
from pathlib import Path

module_path = os.path.abspath(os.path.join("../"))
if module_path not in sys.path:
    sys.path.append(module_path)

from config_loaders.dataset_config_loader import DatasetConfig
from config_loaders.rk_config_loader import RKConfig
from config_loaders.sk_config_loader import SKConfig
from utils.data_io import load_jsonl, load_pickle, make_dir
from utils.vis import save_graph_to_img

warnings.filterwarnings("ignore")

/opt/homebrew/Caskroom/miniforge/base/envs/mllm_diag/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PROMPT_VERSION = 4
MODEL = "internvl2"
DATASET = "vqav2"

In [3]:
rk_config = RKConfig()
sk_config = SKConfig()
ds_config = DatasetConfig()

ds_list = ds_config.get_ds_list()
ds_list.remove("vqav2_holdout")

Load responses_counter
- These files already include the answer for the "complete RK"

In [4]:
# TODO: change this to Path(__file__).parent for script
base_dir = Path("../")
resps_path = rk_config.get_parsed_rk_paths(MODEL, DATASET).format(PROMPT_VERSION)
resps_path = base_dir.joinpath(resps_path)
print(resps_path)
resps = load_jsonl(resps_path)

alt_resps_path = rk_config.get_counterfactual_rk_paths(MODEL, DATASET)
alt_resps_path = base_dir.joinpath(alt_resps_path)
print(alt_resps_path)
alt_resps = load_jsonl(alt_resps_path)

../data/really_know/parsed/internvl2/vqav2/results_v4.jsonl
../data/really_know/counterfactual/internvl2/vqav2/results.jsonl


In [5]:
# Get responses only for the max num of triplets
# (i.e., ignore the rest of the powerset for now)
resps_ids = set([q["question_id"] for q in resps])
alt_resps_ids = set([q["question_id"] for q in alt_resps])

# question_id: response
dict_eval_resp = {
    q_idx: next(r["response"] for r in resps if r["question_id"] == q_idx)
    for q_idx in resps_ids
}

# question_id: response
# this only gets the first response
dict_counter_resp = {
    q_idx: next(r["response"] for r in alt_resps if r["question_id"] == q_idx)
    for q_idx in alt_resps_ids
}

For each question, add a new entry corresponding to the original response.

In [6]:
resps[0].keys()

dict_keys(['question_id', 'response', 'rationales', 'triples', 'triple_objs'])

In [7]:
# Based on the counterfactual responses we have, pick and add the original ones.
all_resps = copy.deepcopy(alt_resps)
for q_idx in alt_resps_ids:
    og_response = next((r for r in resps if r["question_id"] == q_idx), None)
    if og_response:
        all_resps.append(
            {
                "question_id": og_response["question_id"],
                "size_treatment": 0,
                "occluded_concepts": [],
                "response": og_response["response"],
            }
        )
    else:
        print(f"We have an issue with question {q_idx}.")

Create dataframe out of counterfactual data and responses

In [8]:
df = pd.DataFrame(all_resps)
df.drop(["path"], axis=1, inplace=True)

In [9]:
df.head()

,question_id,size_treatment,occluded_concepts,response
0,337814001,1,[28-table],3
1,337814001,1,[6-wall],3
2,337814001,1,[26-chair],3
3,337814001,1,[2-leg],3
4,337814001,1,[1-wall],3


In [10]:
# Create mapping between questions and the biggest size of constraints.
dict_max_triplets = {
    q_idx: max(
        [int(q["size_treatment"]) for q in alt_resps if q["question_id"] == q_idx]
    )
    for q_idx in alt_resps_ids
}

# Make dictionary with {question_id: [concepts]}.
# This is done to extract an actual list of concepts, not any other PD objects.
dict_concepts = {
    q_idx: df[
        (df["question_id"] == q_idx)
        & (df["size_treatment"] == dict_max_triplets[q_idx])
    ]["occluded_concepts"].iloc[0]
    for q_idx in dict_max_triplets
}

Add column ```y``` representing the model answer
- If response is numeric, just use that
- If response is text, create lookup for the results of that question

In [11]:
lookup_y = {}

for idx, row in df.iterrows():
    q_idx = row["question_id"]
    if q_idx not in lookup_y:
        lookup_y[q_idx] = {}

    try:
        y = pd.to_numeric(row["response"])
        lookup_y[q_idx][row["response"]] = y
    except Exception as _:
        # Get all rows with same question ID
        curr_q = df[df["question_id"] == q_idx]
        # Create dict entry with list of unique responses
        lookup_y[q_idx] = {v: k for k, v in enumerate(curr_q["response"].unique())}
        y = lookup_y[q_idx][row["response"]]
    finally:
        df.loc[idx, "y"] = y

In [12]:
df.head()

,question_id,size_treatment,occluded_concepts,response,y
0,337814001,1,[28-table],3,3.0
1,337814001,1,[6-wall],3,3.0
2,337814001,1,[26-chair],3,3.0
3,337814001,1,[2-leg],3,3.0
4,337814001,1,[1-wall],3,3.0


In [13]:
# Make dictionary of dataframes w.r.t. question id
df_dict = {q_idx: df[df["question_id"] == q_idx] for q_idx in dict_counter_resp}

In [14]:
df_dict[399744001].head()

,question_id,size_treatment,occluded_concepts,response,y
111321,399744001,1,[33-man],13,13.0
111322,399744001,1,[7-glass],13,13.0
111323,399744001,1,[5-face],13,13.0
111324,399744001,1,[1-head],13,13.0
111325,399744001,1,[30-man],13,13.0


Let us make binary columns indicating whether a concept was occluded or not.
By default, all concepts are present (1). Then, we zero out the ones occluded.

In [15]:
# Adding 1 column (of zeros) to each sub-dataframe for each concept found
for q_idx in df_dict:
    for concept in dict_concepts[q_idx]:
        df_dict[q_idx][concept] = 1

In [16]:
df_dict[399744001].head()

,question_id,size_treatment,occluded_concepts,response,y,33-man,7-glass,5-face,1-head,30-man,10-hair,31-hat,16-head,11-sunglass,3-face,26-shirt,13-hair,20-head,9-ceiling,6-head
111321,399744001,1,[33-man],13,13.0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
111322,399744001,1,[7-glass],13,13.0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
111323,399744001,1,[5-face],13,13.0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
111324,399744001,1,[1-head],13,13.0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
111325,399744001,1,[30-man],13,13.0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1


In [17]:
df_dict[399744001]["size_treatment"].unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15,  0])

In [18]:
# Correcting the dataframes based on occlusions
for q_idx in df_dict:
    for id, row in df_dict[q_idx].iterrows():
        curr_concepts = row["occluded_concepts"]
        for c in curr_concepts:
            df_dict[q_idx].loc[id, c] = 0
    df_dict[q_idx].drop(["occluded_concepts"], axis=1, inplace=True)

In [19]:
df_dict[399744001].head()

,question_id,size_treatment,response,y,33-man,7-glass,5-face,1-head,30-man,10-hair,31-hat,16-head,11-sunglass,3-face,26-shirt,13-hair,20-head,9-ceiling,6-head
111321,399744001,1,13,13.0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1
111322,399744001,1,13,13.0,1,0,1,1,1,1,1,1,1,1,1,1,1,1,1
111323,399744001,1,13,13.0,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1
111324,399744001,1,13,13.0,1,1,1,0,1,1,1,1,1,1,1,1,1,1,1
111325,399744001,1,13,13.0,1,1,1,1,0,1,1,1,1,1,1,1,1,1,1


```df_dict```
- Dictionary with question_ids as keys for which we have valid RKs
- Each entry in the dictionary is a dataframe with
  - The original response (no occlusion; size_treatment=0)
  - A bunch of rows with the counterfactual responses (combinations of concepts)
  - A series of binary columns matching 1:1 the concepts present or occluded

### Causal Analysis

First, we make create the NX graphs and export the .dot files required by the dowhy package.

Make sure to run ```should_know_processing/pickle_sk.py``` to obtain the pickled files first.

In [20]:
# Load pickled SK graphs
sk_paths = sk_config.get_sk_paths(DATASET)
sk_graphs = {}
all_img_ids = []
for ds_class in sk_paths:
    sk_dir = sk_paths[ds_class]["dir"]
    sk_pkl_path = sk_paths[ds_class]["pkl"]
    curr_graphs = load_pickle(Path("..").joinpath(sk_dir).joinpath(sk_pkl_path))
    all_img_ids.extend(list(curr_graphs.keys()))
    print(f"Loaded {len(curr_graphs)} from {DATASET}+{ds_class}")
    print(f"- Old graph dict: {len(sk_graphs)} entries.")
    sk_graphs.update(curr_graphs)
    print(f"- New graph dict: {len(sk_graphs)} entries.")

Loaded 75 from vqav2+how_many_people_are
- Old graph dict: 0 entries.
- New graph dict: 75 entries.
Loaded 75 from vqav2+what_is_the_person
- Old graph dict: 75 entries.
- New graph dict: 148 entries.


In [21]:
print(f"Duplicate ids: {set([x for x in all_img_ids if all_img_ids.count(x) > 1])}")

Duplicate ids: {'473299', '154520'}


In [68]:
# Add to each graph a node corresponding to the outcome variable 'y'
for sk in sk_graphs.values():
    sk.add_node("y")

    for node in list(sk.nodes()):
        sk.add_edge(node, "y")

In [69]:
# Load question data. Needed to match df_dict and sk_graphs
ds_paths = ds_config.get_ds_paths(DATASET)
questions = []
for ds_class in ds_paths:
    questions_path = Path("..").joinpath(ds_paths[ds_class]["sampled_questions"])
    data = load_jsonl(questions_path)
    questions.extend(data)
print(f"Loaded {len(questions)} questions from {DATASET}.")

Loaded 150 questions from vqav2.


Running causal analysis

In [117]:
def run_causal_inference(data, nx_graph, return_df=False, do_refute=False):
    res_ci = []
    all_estimates = []
    refutations = []

    # Prepare data for multinomial logistic regressor
    responses = data["y"]  # 1-d vector of the responses
    cols_to_skip = ["question_id", "size_treatment", "response"]
    cols_to_keep = list(set(data.keys()) - set(cols_to_skip))
    data_for_estimation = data[cols_to_keep]  # n_observations x regressors
    regressors = data_for_estimation.drop("y", axis=1)

    for col in cols_to_keep:
        if col == "y":
            continue
        print(f"Working on {col}...")

        # effect_modifiers: estimates are separately computed (conditioned)
        # for each value of the effect_modifier.
        model = CausalModel(
            data=data_for_estimation,
            treatment=col,
            outcome="y",
            graph=nx_graph,
            effect_modifiers=list(regressors.keys()),
            missing_nodes_as_confounders=True,
        )
        estimand = model.identify_effect(proceed_when_unidentifiable=True)

        # Control and treatment values are correct since we
        # are occluding (removing) elements from images.
        ce_estimate = model.estimate_effect(
            estimand,
            method_name="backdoor.generalized_linear_model",
            control_value=1,
            treatment_value=0,
            test_significance=True,
            fit_estimator=True,
            target_units=data_for_estimation,
            method_params={"glm_family": sm.families.Binomial()},
        )
        # method_params={"glm_family": sm.families.Binomial()}
        # method_params={"glm_family": sm.MNLogit(responses, regressors)}

        all_estimates.append(ce_estimate)
        res_ci.append({"treatment": col, "ce": str(ce_estimate.value)})

        # Run refutation with Placebo
        if do_refute:
            res_refutation = model.refute_estimate(
                estimand,
                ce_estimate,
                method_name="placebo_treatment_refuter",
                show_progress_bar=False,
                placebo_type="permute",
            )
            refutations.append({col: res_refutation})

    # Sort causal estimates before returning
    sorted_res_ci = sorted(res_ci, key=lambda x: x["ce"], reverse=True)
    if return_df:
        return pd.DataFrame.from_dict(sorted_res_ci), all_estimates, refutations
    else:
        return sorted_res_ci, all_estimates, refutations

In [118]:
res = {}
estimates = {}
refutations = {}

for curr_q in questions:
    curr_idx = curr_q["question_id"]
    print(f">> Working with question # {curr_idx}")
    curr_sk_graph = copy.deepcopy(sk_graphs[curr_q["img"]])
    curr_df = df_dict[curr_idx]

    # removing self loops for causal analysis
    curr_sk_graph.remove_edges_from(nx.selfloop_edges(curr_sk_graph))

    # Save graph image
    # curr_out_dir = f"{CE_OUT_DIR}/{MODEL}/{DATASET}"
    # make_dir(curr_out_dir)
    # save_graph_to_img(g_idx, graph, curr_out_dir)

    # Compute causal effects
    res[curr_idx], estimates[curr_idx], refutations[curr_idx] = run_causal_inference(
        curr_df, curr_sk_graph, do_refute=False
    )

    # Do computations only for 1, for now
    break

>> Working with question # 337814001
Working on 32-glass...
Working on 1-wall...
Working on 15-head...
Working on 2-leg...
Working on 20-pant...
Working on 5-face...
Working on 28-table...
Working on 36-chair...
Working on 14-leg...
Working on 6-wall...
Working on 26-chair...
Working on 35-man...
Working on 37-shoe...
Working on 17-clock...
Working on 18-shirt...
Working on 25-arm...


In [119]:
res

{337814001: [{'treatment': '18-shirt', 'ce': '0.036782538399353304'},
  {'treatment': '36-chair', 'ce': '0.0004042037186742231'},
  {'treatment': '32-glass', 'ce': '0.0'},
  {'treatment': '1-wall', 'ce': '0.0'},
  {'treatment': '15-head', 'ce': '0.0'},
  {'treatment': '2-leg', 'ce': '0.0'},
  {'treatment': '20-pant', 'ce': '0.0'},
  {'treatment': '5-face', 'ce': '0.0'},
  {'treatment': '28-table', 'ce': '0.0'},
  {'treatment': '14-leg', 'ce': '0.0'},
  {'treatment': '6-wall', 'ce': '0.0'},
  {'treatment': '26-chair', 'ce': '0.0'},
  {'treatment': '35-man', 'ce': '0.0'},
  {'treatment': '37-shoe', 'ce': '0.0'},
  {'treatment': '17-clock', 'ce': '0.0'},
  {'treatment': '25-arm', 'ce': '0.0'}]}

In [ ]:
for est in estimates[337814001]:
    print("=" * 30)
    print(est)

*** Causal Estimate ***

## Identified estimand
Estimand type: EstimandType.NONPARAMETRIC_ATE

### Estimand : 1
Estimand name: backdoor
Estimand expression:
     d           
───────────(E[y])
d[32-glass]      
Estimand assumption 1, Unconfoundedness: If U→{32-glass} and U→y then P(y|32-glass,,U) = P(y|32-glass,)

## Realized estimand
b: y~Sigmoid(32-glass+32-glass*1-wall+32-glass*15-head+32-glass*2-leg+32-glass*20-pant+32-glass*37-shoe+32-glass*28-table+32-glass*17-clock+32-glass*36-chair+32-glass*14-leg+32-glass*18-shirt+32-glass*6-wall+32-glass*35-man+32-glass*25-arm+32-glass*26-chair)
Target units: 

## Estimate
Mean value: 0.0
p-value: [0.999, 1]
### Conditional Estimates
__categorical__1-wall  __categorical__15-head  __categorical__2-leg  __categorical__20-pant  __categorical__37-shoe  __categorical__28-table  __categorical__17-clock  __categorical__36-chair  __categorical__14-leg  __categorical__18-shirt  __categorical__6-wall  __categorical__35-man  __categorical__25-arm  __cat

---

Loading pickled graphs to get concept lists

In [ ]:
# NOTE
# These fixes were made to adjust for formatting issues of the models' RKs.
# Not needed now with the graph "variations" being made with the SKs.

# Ad-hoc fixes to comply with pydot formatting
if DATASET == "seed" and MODEL == "LLaVa-1.6":
    # Fix column name 'flagpole, material: wood' for question with ID 4106
    idx_to_fix = "4106"
    df_dict[idx_to_fix].rename(
        {"flagpole, material: wood": "wood"}, axis=1, inplace=True
    )
    # Do the same for the graph file
    og_graphs_dict[idx_to_fix] = nx.relabel_nodes(
        og_graphs_dict[idx_to_fix], {"flagpole, material: wood": "wood"}
    )
    og_graphs_dict[idx_to_fix].nodes["wood"]["label"] = "wood"

    # Fix edge property for question with ID 2740
    idx_to_fix = "2740"
    og_graphs_dict[idx_to_fix]["groceries"]["bottles"]["label"] = "contains"

In [4]:
# TODO: Correct out path
CE_OUT_DIR = "./causal_effects"
make_dir(CE_OUT_DIR)

In [22]:
# Write code to save results to file
for q_idx in res:
    curr_df = pd.DataFrame.from_dict(res[q_idx])
    curr_df.to_csv(f"{curr_out_dir}/ce_{q_idx}.csv")

In [ ]:
# Saving refutation results
out_text = ""
for r in refutations:
    out_text += f">> Refutations results for {r}.\n\n"
    for v in refutations[r]:
        for treatment in v:
            out_text += f'~ Current treatment variable: "{treatment}"\n'
            out_text += f"{v[treatment]}\n"
    out_text += "=" * 30
    out_text += "\n"

with open(f"{curr_out_dir}/refutations_{DATASET}.txt", "w") as fp:
    fp.write(out_text)